In [13]:
#문제 1.
#무신사 사이트 > 랭킹 카테고리에서 상위 50개 상품 정보 수집 후 SQL로 분석
#1) 상품명 / 브랜드명 / 가격 / 할인률 / 리뷰 수
#2) MySQL 저장
#3) SQL로 분석해야 하는 내용
#- 할인률 상위 5개 상품 조회
#- 브랜드별 평균할인률 및 평균 리뷰 수
#- 리뷰수 대비 가격이 가장 높은 상품 TOP 3

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time
import re # 정규 표현식 (숫자만 추출)
from selenium.common.exceptions import NoSuchElementException # 요소가 없을 때 발생하는 예외 처리
import pymysql # MySQL 데이터베이스 연결 라이브러리

# --- MySQL 접속 정보 설정 ---
# ⚠️ 여기에 본인의 MySQL 접속 정보를 정확하게 입력하세요! ⚠️
host = 'localhost'
user = 'root'
password = 'Wonsil929621!@#'
db_name = 'musinsa_ranking' # 데이터를 저장할 데이터베이스 이름 (이미 생성되었다고 가정)
charset = 'utf8'
port = 3306 

# --- Selenium WebDriver 설정 ---
service = Service(ChromeDriverManager().install())
options = Options()
options.add_argument('--window-size=1920x1080')
options.add_argument('--start-maximized')
options.add_argument('--user-agent= Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36')
options.add_argument('--lang=ko_KR')

driver = None # 드라이버 객체 초기화 (finally 블록에서 사용하기 위함)
conn = None   # MySQL 연결 객체 초기화
cursor = None # MySQL 커서 객체 초기화

try:
    # --- Selenium WebDriver 실행 ---
    driver = webdriver.Chrome(service=service, options=options)
    url = 'https://www.musinsa.com/'

    # --- 1. 무신사 웹사이트 크롤링 시작 ---
    driver.get(url)
    # 랭킹 카테고리 클릭
    press_ranking = driver.find_element(By.CSS_SELECTOR, "a.sc-13b3qki-1[data-index='3']")
    press_ranking.click()
    time.sleep(2) # 페이지 로드 대기

    # 페이지 스크롤하여 더 많은 상품 로드
    scroll_pause = 2
    for _ in range(5): # 5번 스크롤하여 대략 50개 상품 로드 목표
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(scroll_pause)

    # 현재 메인 탭 핸들 저장 (새 탭 열고 닫기 위함)
    main_window_handle = driver.current_window_handle

    # 상위 50개 상품 목록 요소 찾기
    products = driver.find_elements(By.CSS_SELECTOR, "div.gtm-view-item-list[data-item-id]")[:50]
    print(f"총 {len(products)}개 상품 정보를 수집합니다.")

    product_data = [] # 최종적으로 MySQL에 저장할 상품 정보 리스트

    # 각 상품에서 기본 정보 (상품명, 브랜드명, 가격, 할인율, 상세 페이지 URL) 추출
    product_info_list_temp = [] # 임시 저장 리스트
    for product in products:
        try:
            name = product.find_element(By.CSS_SELECTOR, "a.gtm-select-item p[data-mds='Typography']").text.strip()
            brand = product.find_element(By.CSS_SELECTOR, "a.gtm-click-brand p[data-mds='Typography']").text.strip()
            
            discount = ""
            try:
                discount_element = product.find_element(By.CSS_SELECTOR, 'span.text-red')
                discount = discount_element.text.strip()
            except NoSuchElementException:
                discount = "0%" # 할인율이 없는 경우 "0%"로 처리

            price = product.find_element(By.CSS_SELECTOR, "div.sc-1t5ihy5-10 span.text-black[data-mds='Typography']").text.strip()
            review_url = product.find_element(By.CSS_SELECTOR, "a.gtm-select-item").get_attribute("href")
            
            product_info_list_temp.append({
                "name": name,
                "brand": brand,
                "discount": discount,
                "price": price,
                "review_url": review_url
            })
        except Exception as e:
            print(f"상품 목록에서 기본 정보 추출 중 오류 발생: {e}")
            continue

    # 각 상품의 상세 페이지로 이동하여 리뷰 수 크롤링 및 최종 데이터 취합
    for item in product_info_list_temp:
        review_count = 0 # 기본 리뷰 수는 0으로 설정
        try:
            # 새 탭으로 상품 상세 페이지 열기
            driver.execute_script("window.open(arguments[0]);", item['review_url'])
            time.sleep(1) # 새 탭이 열리도록 잠시 대기
            
            # 새 탭으로 전환
            new_window_handle = [handle for handle in driver.window_handles if handle != main_window_handle][0]
            driver.switch_to.window(new_window_handle)
            time.sleep(2) # 페이지 로드 대기

            try: # '스냅 · 후기' 버튼 클릭 시도 (리뷰 수를 보기 위해)
                review_button = driver.find_element(By.CSS_SELECTOR, "button[data-button-id='prd_review_tab']")
                review_button.click()
                time.sleep(2) # 리뷰 탭 로드 대기
                
                try: # 리뷰 수 텍스트에서 숫자만 추출
                    review_count_text = driver.find_element(By.CSS_SELECTOR, "span.sc-g3hx4t-3[data-mds='Typography']").text.strip()
                    review_count = int(re.sub(r"[^\d]", "", review_count_text)) if review_count_text else 0
                except NoSuchElementException:
                    # 리뷰 수가 없으면 기본값 0 유지
                    pass
            except NoSuchElementException:
                # '스냅 · 후기' 버튼이 없는 경우에도 기본값 0 유지
                pass
            except Exception as e:
                print(f"리뷰 수 처리 중 예상치 못한 오류 발생 ({item['name']}): {e}")
            
            # 새 탭 닫기 및 원래 메인 탭으로 복귀
            driver.close()
            driver.switch_to.window(main_window_handle)
            time.sleep(1) # 탭 전환 대기
            
            # 수집된 모든 정보 (상품명, 브랜드명, 가격, 할인율, 리뷰수)를 최종 리스트에 추가
            final_item_data = {
                "상품명": item['name'],
                "브랜드명": item['brand'],
                "가격": item['price'],
                "할인률": item['discount'],
                "리뷰수": review_count
            }
            product_data.append(final_item_data)
            print(f"수집 완료: {final_item_data}")
        
        except Exception as e:
            print(f"상품 상세 정보 처리 중 오류 발생 ({item['name']}): {e}")
            continue

    print("\n--- 전체 수집된 상품 정보 ---")
    for i, data in enumerate(product_data, start=1):
        print(f"{i}-{data}")

    # --- 2. 수집된 데이터를 MySQL 데이터베이스에 직접 삽입 ---
    if product_data: # 수집된 상품 데이터가 있을 경우에만 저장 시도
        print("\n--- 수집된 데이터를 MySQL에 저장 중... ---")
        try:
            # MySQL 데이터베이스에 연결
            conn = pymysql.connect(
                host = host,
                port = port,   
                user= user,
                password = password,
                database = db_name, # 정확한 데이터베이스 이름으로 연결
                charset = charset
            )
            cursor = conn.cursor()

            # 데이터 삽입 SQL 쿼리 (여러 행 동시 삽입에 %s 사용)
            sql_insert = """
                INSERT INTO ranking_products (product_name, brand_name, price, discount_rate, review_count)
                VALUES (%s, %s, %s, %s, %s)
            """
            
            insert_values = []
            for item in product_data:
                # 데이터 정제: 가격, 할인율, 리뷰 수에서 숫자만 추출하여 정수형으로 변환
                # .get()을 사용하여 키가 없을 경우 기본값을 제공
                price_val = int(re.sub(r"[^\d]", "", item.get('가격', '0')))
                # 할인율은 '%' 제거 후 숫자로, 없을 경우 0으로
                discount_rate_val = int(re.sub(r"[^\d]", "", item.get('할인률', '0%'))) if item.get('할인률') else 0
                review_count_val = int(item.get('리뷰수', 0))
                
                insert_values.append((item['상품명'], item['브랜드명'], price_val, discount_rate_val, review_count_val))

            # executemany를 사용하여 준비된 모든 데이터를 한 번에 삽입
            cursor.executemany(sql_insert, insert_values)
            conn.commit() # 데이터베이스에 변경사항을 최종 저장 (필수!)
            print(f"{cursor.rowcount}개 상품 데이터가 MySQL에 성공적으로 저장되었습니다. ✅")

        except pymysql.MySQLError as err:
            print(f"MySQL 데이터 삽입 중 오류 발생: {err}")
        
        finally:
            # MySQL 연결을 안전하게 닫습니다.
            if cursor:
                cursor.close()
            if conn and conn.open: # conn 객체가 존재하고 열려있는 경우에만 닫음
                conn.close()
                print("MySQL 연결이 종료되었습니다.")
    else:
        print("수집된 상품이 없어 데이터베이스에 저장할 데이터가 없습니다.")

except Exception as e:
    # 전체 크롤링/저장 과정 중 발생하는 모든 예상치 못한 오류를 처리
    print(f"전체 프로세스 중 치명적인 오류 발생: {e}")

finally:
    # --- 3. 모든 연결 종료 (Selenium 드라이버) ---
    # Selenium 드라이버가 열려있으면 닫아줍니다.
    if driver: # driver 객체가 존재하는지 확인
        driver.quit()
        print("Selenium WebDriver가 종료되었습니다.")


총 50개 상품 정보를 수집합니다.
수집 완료: {'상품명': 'NBXDFA502M / MEN 블랙 코어 에어로쿨 드로즈 5PACK (BLACK)', '브랜드명': '뉴발란스', '가격': '29,940원', '할인률': '40%', '리뷰수': 556}
수집 완료: {'상품명': '젤-1130 - 화이트:퓨어실버 / 1201B020-100', '브랜드명': '아식스', '가격': '119,000원', '할인률': '0%', '리뷰수': 2}
수집 완료: {'상품명': '[2PACK] 아이스센트 크롭 숏슬리브 XFK1TS1001', '브랜드명': '젝시믹스', '가격': '19,800원', '할인률': '66%', '리뷰수': 20}
수집 완료: {'상품명': '에어맥스 뮤즈 W - 블랙:메탈릭 실버 / FV1920-001', '브랜드명': '나이키', '가격': '179,000원', '할인률': '0%', '리뷰수': 165}
수집 완료: {'상품명': 'XT-위스퍼_L47762000', '브랜드명': '살로몬', '가격': '210,000원', '할인률': '0%', '리뷰수': 332}
수집 완료: {'상품명': 'H-Street OG - 블랙:실버 / 403692-02', '브랜드명': '푸마', '가격': '129,000원', '할인률': '0%', '리뷰수': 191}
수집 완료: {'상품명': '데일리 쿨 하프 니트', '브랜드명': '수아레', '가격': '25,000원', '할인률': '49%', '리뷰수': 3265}
수집 완료: {'상품명': '헤링본 볼 캡-그린', '브랜드명': '폴로 랄프 로렌', '가격': '139,000원', '할인률': '0%', '리뷰수': 25}
수집 완료: {'상품명': 'STAR WING SHEER LONG SLEEVE_GY', '브랜드명': '길라아카이브', '가격': '29,500원', '할인률': '50%', '리뷰수': 173}
수집 완료: {'상품명': 'YE OLDE 반스 베이스볼 저지 셔츠 - 

In [12]:
import pymysql
import pandas as pd

from sqlalchemy import create_engine

#데이터베이스 연결 
host = 'localhost'
user = 'root'
password = 'Wonsil929621!@#'
db_name = 'musinsa_ranking'
charset = 'utf8'
port = 3306 

db = pymysql.connect(
    host = host,
    user= user,
    password = password,
    database = db_name
    
)

cursor = db.cursor()


#데이터베이스 생성 
#cursor = db.cursor()
#sql = "CREATE DATABASE IF NOT EXISTS musinsa_ranking"
#cursor.execute(sql)

#테이블 생성 

sql = """ 
    CREATE TABLE IF NOT EXISTS ranking_products (
    id INT AUTO_INCREMENT PRIMARY KEY,
    product_name VARCHAR(150) NOT NULL,
    brand_name VARCHAR(50) NOT NULL,
    price INT NOT NULL,
    discount_rate INT NOT NULL,
    review_count INT NOT NULL   
    )


"""
cursor.execute(sql)
print("테이블 생성완료")
db.commit()


테이블 생성완료


In [ ]:
#문제 2.
#스타벅스 사이트 > 메뉴 > 음료 > 카테고리(선택) 데이터 기반으로 마케팅 인사이트 도출
#1) 원하는 카테고리 선택 후 음료 페이지 크롤링 (이름, 사진, 용량, 칼로리)
#2) 크롤링 데이터 엑셀 출력
#3) SQL로 분석해야 하는 내용
#- 카테고리별 평균 칼로리
#- 상품금액과 칼로리 상관관계 분석